In [ ]:
import os

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import glob
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import numpy as np
np.random.seed(0)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torch.optim.lr_scheduler import MultiStepLR 

import lpips
from pytorch_msssim import SSIM

from models_archs.WaveletFusionNet import WaveletFusionNet
from project_folder.DISTS_pt import DISTS
from degradations.degradation import create_degradation_pipeline

from project_folder.best_model2 import Vision2

def seed_everything(seed=42):
    import random
    import numpy as np
    import torch
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
        
CONFIG = {
    'total_iterations': 400000,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'patch_size': 256,
    'val_interval': 5000,
    'val_batches': 100,
    'num_workers': 8,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'plot_path': 'loss_graph/vision2_curve.png',
    'lr_gamma': 0.5
}

os.makedirs(os.path.dirname(CONFIG['plot_path']), exist_ok=True)


class RestorationDataset(Dataset):
    def __init__(self, image_paths, patch_size=256):
        self.patch_size = patch_size
        self.image_paths = image_paths
        self.degradation_transform = create_degradation_pipeline()
        self.to_tensor = transforms.ToTensor()
        self.crop = transforms.RandomCrop(patch_size)
        
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        gt_img = Image.open(img_path).convert('RGB')
        
        if gt_img.size[0] < self.patch_size or gt_img.size[1] < self.patch_size:
            pad_transform = transforms.Pad((0, 0, max(0, self.patch_size - gt_img.size[0]), 
                                                  max(0, self.patch_size - gt_img.size[1])))
            gt_img = pad_transform(gt_img)
            
        gt_patch = self.crop(gt_img)
        gt_patch_np = np.array(gt_patch)

        augmented = self.degradation_transform(image=gt_patch_np)
        lq_patch_np = augmented['image']
        
        lq_patch = self.to_tensor(lq_patch_np)
        gt_patch = self.to_tensor(gt_patch)
            
        return lq_patch, gt_patch


class BenchmarkDataset(Dataset):
    def __init__(self, root_dir):
        self.clean_dir = os.path.join(root_dir, 'clean')
        self.degraded_dir = os.path.join(root_dir, 'degraded')
        
        self.filenames = sorted([f for f in os.listdir(self.clean_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        
        gt_img = Image.open(os.path.join(self.clean_dir, fname)).convert('RGB')
        lq_img = Image.open(os.path.join(self.degraded_dir, fname)).convert('RGB')
        
        return self.to_tensor(lq_img), self.to_tensor(gt_img)

def create_val_benchmark(val_source_paths, output_dir, num_patches=10000, patch_size=256):
    if os.path.exists(output_dir):
        print(f"--> Валидационный бенчмарк '{output_dir}' обнаружен. Генерация не требуется.")
        return 
        
    clean_dir = os.path.join(output_dir, 'clean')
    degraded_dir = os.path.join(output_dir, 'degraded')

    os.makedirs(clean_dir, exist_ok=True)
    os.makedirs(degraded_dir, exist_ok=True)
    
    transform = create_degradation_pipeline()
    print(f"--> Генерируем {num_patches} валидационных патчей в {output_dir}...")
    
    pbar = tqdm(total=num_patches)
    count = 0
    while count < num_patches:
        img_path = random.choice(val_source_paths)
        
        try:
            img_pil = Image.open(img_path).convert('RGB')
        except:
            continue
            
        w, h = img_pil.size
        if w < patch_size or h < patch_size:
            continue
            
        left = random.randint(0, w - patch_size)
        top = random.randint(0, h - patch_size)
        clean_patch_pil = img_pil.crop((left, top, left + patch_size, top + patch_size))
        
        clean_patch_np = np.array(clean_patch_pil)
        
        degraded_patch_np = transform(image=clean_patch_np)['image']
        
        fname = f"val_{count:05d}.png"
        clean_patch_pil.save(os.path.join(clean_dir, fname))
        Image.fromarray(degraded_patch_np).save(os.path.join(degraded_dir, fname))
        
        count += 1
        pbar.update(1)
    pbar.close()

def calculate_psnr_batch(output, target):
    mse = torch.mean((output - target) ** 2, dim=[1, 2, 3])
    psnr = 10 * torch.log10(1.0 / (mse + 1e-10))
    return torch.mean(psnr).item()

def save_metrics_plot(iterations, metrics_dict, path):
    plt.figure(figsize=(14, 10))
    
    plt.subplot(2, 2, 1)
    plt.plot(iterations, metrics_dict['PSNR'], color='tab:blue')
    plt.xlabel('Iterations')
    plt.ylabel('dB')
    plt.title('PSNR (Higher is better)')
    plt.grid(True)
    
    plt.subplot(2, 2, 2)
    plt.plot(iterations, metrics_dict['SSIM'], color='tab:green')
    plt.xlabel('Iterations')
    plt.title('SSIM (Higher is better)')
    plt.grid(True)
    
    plt.subplot(2, 2, 3)
    plt.plot(iterations, metrics_dict['LPIPS'], color='tab:red')
    plt.xlabel('Iterations')
    plt.title('LPIPS (Lower is better)')
    plt.grid(True)
    
    plt.subplot(2, 2, 4)
    plt.plot(iterations, metrics_dict['DISTS'], color='tab:orange')
    plt.xlabel('Iterations')
    plt.title('DISTS (Lower is better)')
    plt.grid(True)
        
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.close()
    
class NulltoZeroLoss(nn.Module):
    def __init__(self, device):
        super(NulltoZeroLoss, self).__init__()
        self.l1 = nn.L1Loss()
        self.ssim_module = SSIM(data_range=1.0, size_average=True, channel=3)
        self.lpips = lpips.LPIPS(net='alex').to(device)
        self.dists = DISTS().to(device)
        
    def forward(self, pred, target):
        l1_loss = self.l1(pred, target)
        
        ssim_val = self.ssim_module(pred, target)
        ssim_loss = 1.0 - ssim_val
        
        pred_lpips = pred.clamp(0.0, 1.0) * 2.0 - 1.0
        target_lpips = target.clamp(0.0, 1.0) * 2.0 - 1.0
            
        loss_lpips_val = torch.mean(self.lpips(pred_lpips, target_lpips))

        dists_loss = self.dists(pred, target, require_grad=True, batch_average=True)
        
        total_loss = l1_loss + 0.1 * ssim_loss + loss_lpips_val + dists_loss
        
        return total_loss

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
from pytorch_optimizer.base.type import OPTIMIZER_INSTANCE_OR_CLASS
from pytorch_optimizer.base.type import State
from pytorch_optimizer.base.type import Closure
from pytorch_optimizer.base.type import ParamGroup
from pytorch_optimizer.base.type import Loss
from pytorch_optimizer.base.optimizer import BaseOptimizer
from pytorch_optimizer import SOAP
from collections import defaultdict

class ScheduleFreeWrapper(BaseOptimizer):
    def __init__(
        self,
        optimizer: OPTIMIZER_INSTANCE_OR_CLASS,
        momentum: float = 0.9,
        weight_decay: float = 0.0,
        r: float = 0.0,
        weight_lr_power: float = 2.0,
        maximize: bool = False,
        decoupling_c: float = 200.0,
        **kwargs,
    ):
        self.validate_range(momentum, 'momentum', 0.0, 1.0, '[)')
        self.validate_non_negative(weight_decay, 'weight_decay')
        self.validate_non_negative(decoupling_c, 'decoupling_c')

        self.momentum = momentum
        self.weight_decay = weight_decay
        self.r = r
        self.weight_lr_power = weight_lr_power
        self.train_mode: bool = False
        self.maximize = maximize
        self.decoupling_c = decoupling_c

        self.optimizer: Optimizer = self.load_optimizer(optimizer, **kwargs)

        self._optimizer_step_pre_hooks: Dict[int, Callable] = {}
        self._optimizer_step_post_hooks: Dict[int, Callable] = {}

        self.state: State = defaultdict(dict)
        self.defaults: Defaults = self.optimizer.defaults

    def __str__(self) -> str:
        return 'ScheduleFree'

    @property
    def param_groups(self):
        return self.optimizer.param_groups

    def __getstate__(self):
        return {'state': self.state, 'optimizer': self.optimizer}

    def add_param_group(self, param_group):
        return self.optimizer.add_param_group(param_group)

    def state_dict(self) -> State:
        return {'schedulefree_state': self.state, 'base_optimizer': self.optimizer.state_dict()}

    def load_state_dict(self, state: State) -> None:
        self.state = state['schedulefree_state']
        self.optimizer.load_state_dict(state['base_optimizer'])

    def zero_grad(self, set_to_none: bool = True) -> None:
        self.optimizer.zero_grad(set_to_none)

    @torch.no_grad()
    def eval(self):
        if not self.train_mode:
            return

        for group in self.param_groups:
            for p in group['params']:
                state = self.state[p]
                if 'z' in state:
                    p.lerp_(end=state['z'], weight=1.0 - 1.0 / self.momentum)

        self.train_mode = False

    @torch.no_grad()
    def train(self):
        if self.train_mode:
            return

        for group in self.param_groups:
            for p in group['params']:
                state = self.state[p]
                if 'z' in state:
                    p.lerp_(end=state['z'], weight=1.0 - self.momentum)

        self.train_mode = True

    def init_group(self, group: ParamGroup, **kwargs) -> None:
        if 'step' not in group:
            group['step'] = 0

        for p in group['params']:
            if p.grad is None:
                continue

            grad = p.grad
            if grad.is_sparse:
                raise NoSparseGradientError(str(self))

            state = self.state[p]

            if 'z' not in state:
                state['z'] = p.clone()

    @staticmethod
    def swap(x: torch.Tensor, y: torch.Tensor) -> None:
        x.view(torch.uint8).bitwise_xor_(y.view(torch.uint8))
        y.view(torch.uint8).bitwise_xor_(x.view(torch.uint8))
        x.view(torch.uint8).bitwise_xor_(y.view(torch.uint8))

    @torch.no_grad()
    def step(self, closure: Closure = None) -> Loss:
        if not self.train_mode:
            raise ValueError('optimizer was not in train mode when step is called. call .train() before training')

        loss: Loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            self.init_group(group)
            group['step'] += 1

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad

                self.maximize_gradient(grad, maximize=self.maximize)

                state = self.state[p]

                z = state['z']

                self.apply_weight_decay(
                    z,
                    grad=grad,
                    lr=group['lr'],
                    weight_decay=self.weight_decay,
                    weight_decouple=True,
                    fixed_decay=False,
                )

                self.apply_weight_decay(
                    p,
                    grad=grad,
                    lr=group['lr'],
                    weight_decay=self.weight_decay,
                    weight_decouple=True,
                    fixed_decay=False,
                    ratio=1.0 - self.momentum,
                )

                p.lerp_(end=z, weight=1.0 - 1.0 / self.momentum)

                self.swap(z, p)

        self.optimizer.step()

        for group in self.param_groups:
            lr: float = group['lr'] * group.get('d', 1.0)
            lr_max = group['lr_max'] = max(lr, group.get('lr_max', 0))

            weight: float = (group['step'] ** self.r) * (lr_max ** self.weight_lr_power)
            weight_sum = group['weight_sum'] = group.get('weight_sum', 0.0) + weight

            checkpoint: float = weight / weight_sum if weight_sum != 0.0 else 0.0

            if self.decoupling_c > 0:
                beta1 = group['betas'][0]
                checkpoint = min(1.0, checkpoint * (1.0 - beta1) * self.decoupling_c)

            for p in group['params']:
                if p.grad is None:
                    continue

                state = self.state[p]

                z = state['z']

                self.swap(z, p)

                p.lerp_(end=z, weight=checkpoint)

                p.lerp_(end=state['z'], weight=1.0 - self.momentum)

        return loss

In [ ]:
def main():
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)

    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]

    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)

    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")

    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision2().to(CONFIG['device'])

    optimizer = ScheduleFreeWrapper(SOAP(model.parameters()))
    
    criterion = NulltoZeroLoss(device=CONFIG['device'])

    current_iter = 0
    
    history = {
        'PSNR': [],
        'SSIM': [],
        'LPIPS': [],
        'DISTS': []
    }
    iter_history = []
    
    model.train()
    optimizer.train()

    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq) + lq
            
            loss = criterion(output, gt)

            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 

            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                optimizer.eval()
                
                metrics_accum = {'PSNR': 0.0, 'SSIM': 0.0, 'LPIPS': 0.0, 'DISTS': 0.0}
                batch_count = 0
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq) + val_lq
                        
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        batch_psnr = calculate_psnr_batch(val_output, val_gt)
                        metrics_accum['PSNR'] += batch_psnr
                        
                        batch_ssim = criterion.ssim_module(val_output, val_gt).item()
                        metrics_accum['SSIM'] += batch_ssim
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                        
                        batch_dists = criterion.dists(val_output, val_gt, require_grad=False, batch_average=True).item()
                        metrics_accum['DISTS'] += batch_dists
                        
                        batch_count += 1
                
                denom = max(batch_count, 1)
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                    
                iter_history.append(current_iter)
                
                save_metrics_plot(iter_history, history, CONFIG['plot_path'])
                
                torch.save(model.state_dict(), f"loss_graph/vision2_iter_{current_iter}.pth")
                
                model.train()
                optimizer.train()

    torch.save(model.state_dict(), "final_vision2.pth")

if __name__ == "__main__":
    main()